# Multimodal Meme Understanding: Image, Text, or Both?
### Deep Learning Final Project
Dataset: Memotion 7k (SemEval-2020 Task 8, Task A: 3-class sentiment).

Pipeline (see `docs/project-roadmap.md`):
1. Environment setup & Google Drive mount
2. Dataset download & verification
3. Data preparation & leakage control (Hold-out split)
4. Text-only Baseline (BERT) - Role B
5. Image-only Baseline (ResNet50) - Role C
6. Multimodal Fusion (Concat & Cross-Attention) - Role D
7. Evaluation, ablation & error analysis - Role E & F
8. Demo inference

## 1. Environment Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

BRANCH = os.environ.get('BRANCH', 'main')
REPO_URL = 'https://github.com/arcrek/dl2026.git'
ROOT = '/content/drive/MyDrive/dl2026_memotion'
os.environ['DATA_DIR'] = '/content/data/memotion'
os.environ['FEATURE_DIR'] = f'{ROOT}/features'
os.environ['RESULTS_DIR'] = f'{ROOT}/results'

for d in ('features', 'results'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

if os.path.exists('/content/repo'):
    !git -C /content/repo fetch && git -C /content/repo checkout $BRANCH && git -C /content/repo pull
else:
    !git clone -b $BRANCH $REPO_URL /content/repo
%cd /content/repo

!pip install -q transformers scikit-learn pyarrow huggingface_hub pytest


## 2. Dataset Download & Verification
> **Note:** For the Text-only Baseline (Role B), you can pass `--jsonl-only` to `scripts/download_data.sh` to download text metadata in seconds without fetching 1.5GB of images.

In [ ]:
TAR = f'{ROOT}/memotion.tar'
if os.path.exists(TAR) and not os.path.exists('/content/data/memotion/train.jsonl'):
    !mkdir -p /content/data && tar -xf "$TAR" -C /content/data

!bash scripts/download_data.sh
!python src/data.py --images

if not os.path.exists(TAR):
    !tar -cf "$TAR" -C /content/data memotion


## 3. Data Preparation & Leakage Control

In [ ]:
import json, os, sys
from collections import Counter
sys.path.insert(0, 'src')
import data

D, F = data.DATA_DIR, data.FEATURE_DIR
splits = {s: data.load_jsonl(s, D) for s in data.SPLITS}
train = splits['train']
by_id = {r['id']: r for r in train}

fit_ids, hold_ids = data.save_holdout(D)
with open(os.path.join(F, 'train_holdout.json')) as f:
    meta = json.load(f)
print(f"grouping={meta['grouping']}  fit={len(fit_ids)}  holdout={len(hold_ids)}")
print('class_weights (fit):', [round(w, 3) for w in meta['class_weights']])
for name, ids in (('fit', fit_ids), ('holdout', hold_ids)):
    c = Counter(by_id[i]['label'] for i in ids)
    print(f"{name:8s}", {data.CLASS_NAMES[k]: round(c[k] / len(ids), 3) for k in range(3)})

gid, _ = data.group_ids(train, D)
overlap = {gid[i] for i in fit_ids} & {gid[i] for i in hold_ids}
assert not overlap, f'{len(overlap)} groups straddle fit/holdout'
assert not set(fit_ids) & set(hold_ids)

tr_txt = {data._norm_text(r['text']) for r in train} - {''}
for s in ('validation', 'test'):
    dup = sum(data._norm_text(r['text']) in tr_txt for r in splits[s])
    print(f"{s}: {dup}/{len(splits[s])} captions also appear in train")
assert all('text_ocr' not in r for r in train), 'text_ocr must not be loaded'
print('Leakage checks passed: 0 groups shared between fit and hold-out.')


## 4. Text-only Baseline (BERT) - Role B
Fine-tune `bert-base-uncased` with Differential Learning Rate (`1.5e-5` for backbone, `5e-4` for head) across seeds 0, 1, 2. Model selection uses hold-out Macro-F1, and test predictions are exported to `results/text_seed{0,1,2}.json`.

In [ ]:
!python src/text.py --seeds 0 1 2 --epochs 5 --batch 32


In [ ]:
!pytest tests/test_results.py -v


In [ ]:
import os, sys
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
from results import load_results

records = []
for s in [0, 1, 2]:
    try:
        res = load_results('text', s)
    except FileNotFoundError:
        continue
    records.append({
        'Seed': s,
        'Best Epoch (Holdout)': res['extra'].get('best_epoch', 'N/A'),
        'Holdout Macro-F1': round(res['extra'].get('holdout_macro_f1', 0.0), 4),
        'Test Accuracy': round(res['accuracy'], 4),
        'Test Macro-F1': round(res['macro_f1'], 4)
    })

if records:
    df = pd.DataFrame(records)
    if len(records) > 1:
        mean_row = pd.DataFrame([{
            'Seed': 'Mean +/- Std',
            'Best Epoch (Holdout)': '-',
            'Holdout Macro-F1': f"{df['Holdout Macro-F1'].mean():.4f} +/- {df['Holdout Macro-F1'].std():.4f}",
            'Test Accuracy': f"{df['Test Accuracy'].mean():.4f} +/- {df['Test Accuracy'].std():.4f}",
            'Test Macro-F1': f"{df['Test Macro-F1'].mean():.4f} +/- {df['Test Macro-F1'].std():.4f}"
        }])
        summary_df = pd.concat([df, mean_row], ignore_index=True)
    else:
        summary_df = df
    display(summary_df)
else:
    print('No results found. Run training cell above first.')


In [ ]:
try:
    from google.colab import files
    for s in [0, 1, 2]:
        f_path = f"results/text_seed{s}.json"
        if os.path.exists(f_path):
            files.download(f_path)
except Exception:
    pass


## 5. Image-only Baseline (ResNet50)


## 6. Multimodal Fusion (Concat, Cross-Attention & Product) - Role D
Huấn luyện 3 cấu hình Đa phương thái (Both): Concat Fusion (3A), Cross-Attention Fusion (3B), và Product Fusion (Ablation) qua 3 seeds [0, 1, 2].
Model selection sử dụng hold-out Macro-F1, kết quả suy luận trên test split được lưu chuẩn vào `results/both_*.json`.

In [ ]:
# Train all 3 fusion architectures across seeds 0, 1, 2
!python src/fusion.py --config all --seeds 0 1 2 --epochs 5 --batch 16


In [ ]:
!pytest tests/test_results.py tests/test_fusion.py -v


In [ ]:
import os, sys, json
sys.path.insert(0, 'src')
import numpy as np
from results import load_results

configs = ['both_concat', 'both_cross_attn', 'both_product']
seeds = [0, 1, 2]

print(f"{'Configuration':<20} {'Macro-F1 (mean ± std)':<25} {'Accuracy (mean ± std)':<25}")
print('-' * 70)
for cfg in configs:
    f1s, accs = [], []
    for s in seeds:
        try:
            r = load_results(cfg, s)
            f1s.append(r['macro_f1'])
            accs.append(r['accuracy'])
        except Exception:
            pass
    if f1s:
        print(f"{cfg:<20} {np.mean(f1s):.4f} ± {np.std(f1s):.4f}{'':<10} {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    else:
        print(f"{cfg:<20} Pending training")


## 7. Evaluation, Ablation & Error Analysis


## 8. Demo Inference
